In [1]:
import sys, math, os
import logging
import numpy as np
from ase.io import read, write
from os import environ, listdir, mkdir, chdir
from ase.units import Hartree, Bohr
from os import path
from ase import Atoms
import json
from ase.io import read
import h5py

In [6]:
EPBE0_atom = {1:-13.64136507, 6:-1027.59344601, \
              7:-1484.25358313, 8:-2039.74560236, 9: -3139.751675679, 15:-9283.01120606, 16: -10828.726222083, 17: -12516.462339357}

filtered_dir = "/data/horse/ws/miha493e-research_project/activediffusion/datasets/iter_0/filtered"         # input directory with filtered molecules
h5file_curr = "/data/horse/ws/miha493e-research_project/activediffusion/datasets/iter_0/hdf5/test_output.hdf5"            # output file path of current iteration
h5file_full = "/data/horse/ws/miha493e-research_project/activediffusion/datasets/hdf5/test_qm7x-aqm-gas-noP-output.hdf5"          # output file path of full generated dataset
itr = 0

In [7]:
folders = sorted(listdir(filtered_dir))


In [14]:
with h5py.File(h5file_curr, 'w') as iter_h5file, h5py.File(h5file_full, 'a') as full_h5file:
    for hdf5_file in [iter_h5file, full_h5file]:

        for fdir in folders:

            print(fdir)
            grp = hdf5_file.create_group(f"{fdir}_{itr}")
            molecule = {}

            #folders2 = sorted(listdir(xyz_pre+fdir))
            folder_path = os.path.join(filtered_dir, fdir)
            jfils = [f for f in listdir(folder_path) if f.endswith('.json')]  #As folder have non json files

            for confid in jfils:

                with open(filtered_dir+'/'+str(fdir)+'/'+str(confid)) as json_file:
                    data = json.load(json_file)

                sysname = confid[:-5]

                subgrp = grp.create_group(sysname)
            #saving DFT data

                # w = len(data["Coordinates"])
                # ftmp = open('tmp-'+str(sysname)+'.xyz', 'w')  # open file for writing
                # ftmp.write(str(w) + "\n")
                # ftmp.write("{: >24}".format('leo') + "\n")
                # for p1 in data["Coordinates"]:
                #     ftmp.write("{: >24} {: >24} {: >24} {: >24}".format(*p1) + "\n")
                # ftmp.close()
                #atoms = read('tmp-'+str(sysname)+'.xyz')    # define atoms
                xyz_file_path = os.path.join(folder_path, sysname + '.xyz')       # path to the xyz file if coordinate is not in json file
                atoms = read(xyz_file_path)
                Z = atoms.get_atomic_numbers()
                POS = atoms.get_positions()

                #os.remove('tmp-'+str(sysname)+'.xyz')

                ener=float(data["Total energy (eV):"])
                mbd=float(data["SUM"]["MBD"])
                pbe0 = float(ener) - float(mbd)
                #print("EPBE0_atom keys:", EPBE0_atom.keys())
                #print("Z values:", Z)
                #print([type(zi) for zi in Z])
                EDFTatoms = sum([ EPBE0_atom[zi] for zi in Z ])
                Eat = float(pbe0) - float(EDFTatoms)

                totfor = np.array(data["Total Forces (eV/Ang):"])
                vdwfor = np.array(data["van der Waals Forces (eV/Ang):"])
                pbe0for = totfor - vdwfor

            #Hirshfeld quantities
                HChar = np.array(data["Hirshfeld Charges:"])

            #DIPOLE moment
    #            dipole = float(data["Absolute Total Dipole (eAng):"])
    #            dipcomp = np.array(data["Total Dipole (eAng)"]) 

                FVol = data["Free Atom Volumes:"]
                HVol = data["Hirshfeld Volumes:"]
                hvr = np.divide(HVol,FVol)    # define hirshfeld ratios
                
                molecule['DIP'] = np.array(data["Absolute Total Dipole (eAng):"])
                molecule['HLgap'] = np.array(float(data["HOMO-LUMO gap"]))
                molecule['atNUM'] = np.array(Z)
                molecule['atXYZ'] = np.array(POS)
                molecule['eAT'] = np.array(data["SUM"]["Free atom Elst"]) # or Eat = float(pbe0) - float(EDFTatoms) 
                molecule['eC'] = np.array(data["SUM"]["C energy"])
                molecule['eEE'] = np.array(data["SUM"]["Electrostatic Energy"])
                molecule['eH'] = np.array(data["SUM"]["Hartree-Fock Energy"])
                molecule['eKIN'] = np.array(data["SUM"]["Kinetic Energy"])
                molecule['eMBD'] = np.array(data["SUM"]["MBD"])
                molecule['ePBE0'] = np.array([float(pbe0)])
                molecule['ePBE0+MBD'] = np.array([float(ener)])
                molecule['totFOR'] = totfor
                molecule['pbe0FOR'] = pbe0for
                molecule['eX'] = np.array(data["SUM"]["X energy"])
                molecule['eXC'] = np.array(data["SUM"]["Total XC energy"])
                molecule['eXX'] = np.array(data["SUM"]["Hartree-Fock Energy"])

                for key, value in molecule.items():
                    print(key, value, value.shape, value.dtype)
                    dset = subgrp.create_dataset(key, value.shape, dtype=value.dtype)
                    dset[...] = value

opt-mol_1
DIP 2.16445454353892 () float64
HLgap 1.10823095 () float64
atNUM [ 7 17  6  7 17  6  6  1 17  6  1  1 17  6  1  6  6  1 17  1  1  6  6  6] (24,) int64
atXYZ [[-1.11418239e+00  1.33055468e+00 -1.11206078e-02]
 [ 1.37211880e+00 -1.94527444e-01  4.53776716e+00]
 [-4.36068346e-01  2.00981840e+00 -8.92008107e-01]
 [-3.05439136e-01 -1.16198062e+00 -1.64027418e+00]
 [ 1.22430121e+00  1.16532040e+00 -2.90338798e+00]
 [-1.61312604e+00  6.44938411e-02 -4.91471573e-01]
 [-6.98400874e-01 -1.16588363e+00 -2.19023891e-01]
 [-1.86902924e+00 -1.85334057e-01 -2.72677317e+00]
 [ 2.58541082e-01  3.49387330e+00 -5.45298583e-01]
 [ 4.88153432e-02 -6.85005196e-01  2.15521543e+00]
 [-9.50840046e-01  1.96633542e+00 -2.94554661e+00]
 [-1.29431869e+00 -2.05573605e+00  1.47517237e-02]
 [-1.51421716e+00 -4.27936891e-01  2.78494179e+00]
 [-4.23286350e-01  1.35438477e+00 -2.21335090e+00]
 [-2.67227290e+00 -5.49376408e-02 -2.73029291e-01]
 [-1.12389935e+00  3.01996367e-03 -1.95252732e+00]
 [ 3.38412923e-0